In [15]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_classic.retrievers import EnsembleRetriever
from sentence_transformers import CrossEncoder
from langchain.agents.middleware import wrap_model_call
import subprocess
from pathlib import Path
import sys
from typing import Optional, Literal

load_dotenv()

True

In [2]:
embeddings= HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1455.06it/s]


In [3]:
django_vectorstore= Chroma(
    persist_directory="./django_chroma_db",
    collection_name="django_docs",
    embedding_function=embeddings
)

python_vectorstore= Chroma(
    persist_directory="./django_chroma_db",
    collection_name="python_scripts",
    embedding_function=embeddings
)

In [4]:
django_db= django_vectorstore.get()
python_db=python_vectorstore.get()

django_splits=[
    Document(page_content=text, metadata=meta or {})
    for text, meta in zip(django_db["documents"], django_db["metadatas"])
]

python_splits = [
    Document(page_content=text, metadata=meta or {})
    for text, meta in zip(python_db["documents"], python_db["metadatas"])
]

django_retriever= django_vectorstore.as_retriever(search_kwargs={"k":4})
python_retriever= python_vectorstore.as_retriever(search_kwargs={"k":4})


all_splits= django_splits + python_splits
print(f"Loaded {len(all_splits)} total splits ({len(django_splits)} Django docs + {len(python_splits)} Python codebase).")
bm25_retriever= BM25Retriever.from_documents(all_splits)


Loaded 6361 total splits (5110 Django docs + 1251 Python codebase).


In [5]:
bm25_retriever.k=8

In [6]:
#Creating hybrid retriever
hybrid_retriever= EnsembleRetriever(
    retrievers=[django_retriever, python_retriever, bm25_retriever],
    weights=[0.4,0.3,0.3]#40% django sementic search, 30% python sementic and keyword
    
)

In [7]:
#Reranker block of code
reranker= CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1675.50it/s]


## Generation Tools

In [24]:
DEFAULT_DIR = r"C:\Users\ZENOID\Desktop\Home\home\self_made.projects\AI(created by me) generated projects"

@tool
def retrieve_django_content(query: str) -> str:
    """
    Search the Django knowledge base for documentation, code examples,
    debugging information, and practical examples.
    Use this tool to gain more context and information needed to answer the users question
    """
    print("Using retriever......")
    docs = hybrid_retriever.invoke(query)

    print("Reranking......")
    pairs = [[query, doc.page_content] for doc in docs]
    scores = reranker.predict(pairs)
    scored_docs = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )
    top_docs = scored_docs[:3]

    return "\n\n".join(doc.page_content for doc, score in top_docs)


def _get_bin_dir(env_path: Path) -> Path:
    """Returns the folder inside a venv where executables live (differs by OS)."""
    return env_path / "Scripts" if sys.platform == "win32" else env_path / "bin"


def _exe(bin_dir: Path, name: str) -> Path:
    """Adds .exe to the executable name only on Windows."""
    return bin_dir / (f"{name}.exe" if sys.platform == "win32" else name)


@tool
def setup_django_project(project_name: str,app_name: Optional[list[str] | str] = None,directory: Optional[str] = None) -> dict:
    """
    Set up a Django project safely.
    This operation is idempotent: existing virtual environments,
    Django projects, and apps are detected and will not be recreated.
    """
    if not project_name or not project_name.strip():
        return {"status": "error","message": "project_name is required"}

    if not directory or directory.strip() in ("", "."):
        directory = DEFAULT_DIR

    target_path = Path(directory) / project_name
    target_path.mkdir(parents=True, exist_ok=True)

    results = {
        "status": "success",
        "project_name": project_name,
        "project_path": str(target_path),
        "virtualenv": None,
        "django": None,
        "project": None,
        "apps": []
    }

    # Creating Virtual environment
    env_path = target_path / "env"
    if env_path.exists():
        results["virtualenv"] = "already_exists"
    else:
        print("Creating virtual environent......")
        result = subprocess.run(
            [sys.executable, "-m", "venv", "env"],cwd=target_path,capture_output=True,text=True
        )
        if result.returncode != 0:
            return {
                "status": "error",
                "step": "create_virtualenv",
                "message": result.stderr[-1000:]
            }
        results["virtualenv"] = "created"

    # Getting executables
    bin_dir = _get_bin_dir(env_path)
    python_exe = _exe(bin_dir, "python")
    pip_exe = _exe(bin_dir, "pip")

    # Checking if Django is already installed
    django_check = subprocess.run(
        [str(python_exe), "-c", "import django"],capture_output=True,text=True
    )
    if django_check.returncode == 0:
        results["django"] = "already_installed"
    else:
        print("Installing Django......")
        install_result = subprocess.run(
            [str(pip_exe), "install", "django"],cwd=target_path,capture_output=True,text=True
        )
        if install_result.returncode != 0:
            return {
                "status": "error",
                "step": "install_django",
                "message": install_result.stderr[-1000:]
            }
        results["django"] = "installed"


    # Creating Django Project
    manage_py = target_path / "manage.py"
    project_package = target_path / project_name
    if manage_py.exists() and project_package.exists():
        results["project"] = "already_exists"
    else:
        print("Creating Django project......")
        django_admin = _exe(bin_dir, "django-admin")
        project_result = subprocess.run(
            [str(django_admin),"startproject",project_name,"."],cwd=target_path,capture_output=True,text=True
        )
        if project_result.returncode != 0:
            return {
                "status": "error",
                "step": "create_project",
                "message": project_result.stderr[-1000:]
            }
        results["project"] = "created"


    # Creating Django Apps
    if app_name:
        print("Creating Django Apps......")
        apps = (app_name if isinstance(app_name, list) else [app_name])
        for app in apps:
            app_path = target_path / app
            if app_path.exists():
                results["apps"].append({
                    "name": app,
                    "status": "already_exists"
                })
                continue
            app_result = subprocess.run(
                [str(python_exe),"manage.py","startapp",app],cwd=target_path,capture_output=True,text=True
            )
            if app_result.returncode != 0:
                results["apps"].append({
                    "name": app,
                    "status": "failed",
                    "error": app_result.stderr[-500:]
                })
                results["status"] = "partial_success"
            else:
                results["apps"].append({
                    "name": app,
                    "status": "created"
                })
    return results

_active_server_process = None


@tool
def manage_server(action: str,project_name: Optional[str] = None,directory: Optional[str] = None) -> dict:
    """
    Starts or stops the active Django development server.
    action must be either "start" or "stop".
    For "start", project_name identifies the Django project.
    For "stop", no project_name is required.
    """
    global _active_server_process
    action = action.strip().lower()
    if action not in ("start", "stop"):
        return {
            "status": "error",
            "action": action,
            "message": "Invalid action. Use 'start' or 'stop'."
        }

    if not directory or directory.strip() in ("", "."):
        directory = DEFAULT_DIR

    results = {
        "action": action,
        "project_name": project_name,
        "project_path": None,
        "status": "success"
    }
    # Stops Django Server
    if action == "stop":
        if _active_server_process and _active_server_process.poll() is None:
            _active_server_process.terminate()
            try:
                _active_server_process.wait(timeout=3)
            except subprocess.TimeoutExpired:
                _active_server_process.kill()
            _active_server_process = None
            results["message"] = (
                "Django development server stopped successfully."
            )
        else:
            results["status"] = "not_running"
            results["message"] = (
                "No active Django server is currently running."
            )
        return results


    if not project_name:
        return {
            "status": "error",
            "action": "start",
            "message": "project_name is required when starting the server."
        }

    target_path = Path(directory) / project_name

    # Fallback in case directory itself contains manage.py
    if not (target_path / "manage.py").exists():

        if (Path(directory) / "manage.py").exists():
            target_path = Path(directory)

        else:
            return {
                "status": "error",
                "action": "start",
                "project_name": project_name,
                "message": (
                    f"Could not find manage.py for "
                    f"project '{project_name}'."
                )
            }

    manage_py = target_path / "manage.py"

    results["project_path"] = str(target_path)
    env_path = target_path / "env"
    python_exe = _exe(_get_bin_dir(env_path),"python")

    if not python_exe.exists():

        return {
            "status": "error",
            "action": "start",
            "project_name": project_name,
            "project_path": str(target_path),
            "message": (
                "Could not find the virtual environment Python executable."
            )
        }

    #Checking if Server is running
    if _active_server_process and _active_server_process.poll() is None:
        return {
            "status": "already_running",
            "action": "start",
            "project_name": project_name,
            "project_path": str(target_path),
            "message": (
                "A Django development server is already running."
            )
        }

    #Start Django Server

    try:
        _active_server_process = subprocess.Popen([str(python_exe),"manage.py","runserver","--noreload"],cwd=target_path,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
        results["status"] = "started"
        results["message"] = f"Django server started successfully for project '{project_name}'."
        return results
    except Exception as error:
        return {
            "status": "error",
            "action": "start",
            "project_name": project_name,
            "project_path": str(target_path),
            "message": str(error)
        }

@tool
def run_django_commands(
    command: Literal["makemigrations", "migrate", "collectstatic", "test", "flush", "check"],
    project_name: Optional[str] = None,
    directory: Optional[str] = None,
    app_label: Optional[str] = None,
    no_input: bool = False,
    verbose: bool = False
) -> dict:
    """
    Run Django management commands on a Django project.
    Supported commands: makemigrations, migrate, collectstatic, test, flush, check.
    
    Parameters:
    - command: The Django management command to run
    - project_name: Name of the Django project (if not provided, uses directory itself)
    - directory: Directory containing the project (defaults to DEFAULT_DIR)
    - app_label: Specific app to target (for makemigrations and migrate)
    - no_input: Skip prompts, use defaults (for collectstatic and flush)
    - verbose: Increase verbosity of output (for test, migrate, etc.)
    
    Returns: Command output and status
    """
    print(f"Running Django command: {command}......")
    
    if not directory or directory.strip() in ("", "."):
        directory = DEFAULT_DIR
    
    command = command.strip().lower()
    
    # If project_name is provided, use it; otherwise try current directory
    if project_name:
        target_path = Path(directory) / project_name
    else:
        target_path = Path(directory)
    
    # Fallback: if manage.py doesn't exist at target_path, try directory itself
    if not (target_path / "manage.py").exists():
        if (Path(directory) / "manage.py").exists():
            target_path = Path(directory)
        else:
            return {
                "status": "error",
                "error_code": "MANAGE_PY_NOT_FOUND",
                "project_name": project_name,
                "directory": str(target_path),
                "message": f"Could not find manage.py for project '{project_name}' at {target_path}"
            }
    
    # Get Python executable from venv
    env_path = target_path / "env"
    if not env_path.exists():
        return {
            "status": "error",
            "error_code": "VENV_NOT_FOUND",
            "project_name": project_name,
            "project_path": str(target_path),
            "message": "Virtual environment not found. Cannot run commands."
        }
    
    bin_dir = _get_bin_dir(env_path)
    python_exe = _exe(bin_dir, "python")
    
    if not python_exe.exists():
        return {
            "status": "error",
            "error_code": "PYTHON_EXE_NOT_FOUND",
            "project_name": project_name,
            "project_path": str(target_path),
            "message": "Could not find Python executable in virtual environment"
        }
    
    # Build command with whitelisted, hardcoded flags only
    cmd_list = [str(python_exe), "manage.py", command]
    
    # Add command-specific flags based on parameters
    if command == "makemigrations":
        if app_label:
            cmd_list.extend(["--app", app_label])
        if verbose:
            cmd_list.append("--verbose=2")
    
    elif command == "migrate":
        if app_label:
            cmd_list.append(app_label)
        if verbose:
            cmd_list.append("--verbose=2")
    
    elif command == "collectstatic":
        if no_input:
            cmd_list.append("--noinput")
        if verbose:
            cmd_list.append("--verbose=2")
    
    elif command == "test":
        if app_label:
            cmd_list.append(app_label)
        if verbose:
            cmd_list.append("--verbose=2")
    
    elif command == "flush":
        if no_input:
            cmd_list.append("--noinput")
    
    elif command == "check":
        if verbose:
            cmd_list.append("--verbose=2")
    
    try:
        result = subprocess.run(
            cmd_list,
            cwd=target_path,
            capture_output=True,
            text=True
        )
        
        if result.returncode == 0:
            return {
                "status": "success",
                "command": command,
                "project_name": project_name,
                "project_path": str(target_path),
                "output": result.stdout if result.stdout else "Command executed successfully",
                "message": f"Django command '{command}' completed successfully"
            }
        else:
            return {
                "status": "error",
                "error_code": "COMMAND_FAILED",
                "command": command,
                "project_name": project_name,
                "project_path": str(target_path),
                "output": result.stderr[-1000:] if result.stderr else "Unknown error",
                "message": f"Django command '{command}' failed"
            }
    
    except Exception as error:
        return {
            "status": "error",
            "error_code": "COMMAND_EXECUTION_FAILED",
            "command": command,
            "project_name": project_name,
            "project_path": str(target_path),
            "message": str(error)
        }


## File Editor Tools
Prompt/Task -> Search -> Create/Read -> Edit -> Write

In [19]:

@tool
def search_codebase(query: str,directory: Optional[str] = None,search_scope: Literal["files", "folders"] = "files",) -> dict:
    """
    Search a Django codebase for a query.
    Use search_scope="files" to search Python file contents.
    Use search_scope="folders" to search folder names.
    Returns matching paths and relevant line information
    without returning entire file contents.
    """
    print("Enabling codebase search...")

    if not directory or directory.strip() in ("", "."):
        directory = DEFAULT_DIR

    if not query or not query.strip():
        return {
            "status": "error",
            "error_code": "QUERY_REQUIRED",
            "message": "A search query is required."
        }

    query = query.strip()
    path = Path(directory)

    if not path.exists():
        return {
            "status": "error",
            "error_code": "DIRECTORY_NOT_FOUND",
            "directory": str(path),
            "message": f"Directory does not exist: {path}"
        }

    if not path.is_dir():
        return {
            "status": "error",
            "error_code": "INVALID_DIRECTORY",
            "directory": str(path),
            "message": f"The provided path is not a directory: {path}"
        }

    if search_scope == "files":
        print("Searching file contents...")
        matches = []
        for file in path.rglob("*.py"):
            # Ignore virtual environments and dependencies
            if any(excluded in file.parts for excluded in ("env", "venv", ".venv", "site-packages")):
                continue
            try:
                lines = file.read_text(encoding="utf-8").splitlines() #Getting the exact line number of the given/ found query 
            except (UnicodeDecodeError, PermissionError, OSError):
                continue

            #Matching the query with the line number of the given query
            for line_number, line in enumerate(lines, start=1):
                if query.lower() in line.lower():
                    #Appending a list of dictonaries containing the path, line number, and preview of the line
                    matches.append({"path": str(file),"line": line_number,"preview": line.strip()})
        return {
            "status": "success",
            "query": query,
            "directory": str(path),
            "search_scope": "files",
            "matches": matches,
            "total_matches": len(matches)
        }

    elif search_scope == "folders":
        print("Searching folders...")
        matches = []
        for folder in path.rglob("*"):
            if not folder.is_dir():
                continue
            if any(
                excluded in folder.parts
                for excluded in ("env", "venv", ".venv", "site-packages")
            ):
                continue
            if query.lower() in folder.name.lower():
                matches.append({"path": str(folder),"name": folder.name})
        return {
            "status": "success",
            "query": query,
            "directory": str(path),
            "search_scope": "folders",
            "matches": matches,
            "total_matches": len(matches)
        }





@tool
def file_operations(file_path: str, 
                    operation: Literal["read", "edit", "append", "write"],
                    old_code: Optional[str] = None,
                    new_code: Optional[str] = None,
                    overwrite: Optional[bool] = False,
                    start_line: Optional[int]= None,
                    end_line: Optional[int]= None,
                    ) -> dict:
    """
    Perform operations on a file.
    Supported operations:
    read:
        Reads a file. Can read the entire file or a specific
        range of lines using start_line and end_line.
    edit:
        Replaces one exact and unique block of old_code with new_code.
    append:
        Adds new_code to the end of an existing file.
    write:
        Creates a new file with new_code.
        Existing files are not overwritten unless overwrite=True.
    """

    print(f"Performing '{operation}' on {file_path}.......")

    path= Path(file_path)

    operation= operation.strip().lower()

    #Read Operation

    if operation == "read":
        if not path.exists():
            return {
                "status": "error",
                "error_code": "FILE_NOT_FOUND",
                "path": str(path),
            }
        
        try:
            lines= path.read_text(encoding="utf-8").splitlines()
            total_lines= len(lines) #Checking for the total number of files available

            #If theres no line provided for us to start reading we will start from the begining
            if start_line is None:
                start_line= 1

            #Read to the end of the file
            if end_line is None:
                end_line=total_lines

            #Checking if start_line is valid
            if start_line < 1:
                return{
                    "status": "error",
                    "error_code": "INVALID_START_LINE",
                    "path": str(path),
                    "message": "start_line must be greater than or equal to 1"
                }

            #Checking if end_line is valid
            if end_line < start_line:
                return{
                    "status": "error",
                    "error_code": "INVALID_LINE_RANGE",
                    "path": str(path),
                    "message": "end_line cannot be lower than the start_line"
                }

            #Checking if start_line exists
            if start_line > total_lines:
                return{
                    "status": "error",
                    "error_code": "START_LINE_OUT_OF_RANGE",
                    "path": str(path),
                    "total_lines": total_lines,
                }

            #Prevents end_line from going beyond the file
            end_line= min(end_line, total_lines)

            #Minusing it by one because we didnt start from 0 when labelling each match
            selected_lines= lines[start_line -1: end_line]

            #Returnong the line number and the line we read back to the LLM
            content= "\n".join(f"{line_number}: {line}" for line_number, line in enumerate(selected_lines, start=start_line))

            return{
                "status": "success",
                "operation": "read",
                "path": str(path),
                "start_line": start_line,
                "end_line": end_line,
                "total_lines": total_lines,
                "content": content
            }

        except (UnicodeDecodeError, PermissionError, OSError) as error:
            return{
                "status": "error",
                "error_code": "FILE_READ_FAILED",
                "path": str(path),
                "message": str(error)
            }


    #Edit Operation
    elif operation == "edit":
        if not path.exists():
            return{
                "status": "error",
                "error_code": "FILE_NOT_FOUND",
                "path": str(path),
            }
        if not old_code:
            return{
                "status": "error",
                "error_code": "OLD_CODE_REQUIRED",
                "path": str(path),
            }

        if new_code is None:
            return{
                "status": "error",
                "error_code": "NEW_CODE_REQUIRED",
                "path": str(path),
            }
        
        try:
            content= path.read_text(encoding="utf-8")

            count= content.count(old_code)

            if count == 0:
                return {
                    "status": "error",
                    "error_code": "OLD_CODE_NOT_UNIQUE",
                    "path": str(path),
                    "matches": count
                }

            updated_content= content.replace(old_code, new_code, 1)

            path.write_text(updated_content, encoding="utf-8")
            return{
                "status": "modified",
                "operation": "edit",
                "path": str(path),
                "changes_summary": "Successfully replaced the specified code block"
            }
        except (UnicodeDecodeError, PermissionError, OSError) as error:
            return{
                "status": "error",
                "error_code": "FILE_EDIT_FAILED",
                "path": str(path),
                "message": str(error)
            }

    #Append Operation
    elif operation == "append":
        if not path.exists():
            return{
                "status": "error",
                "error_code": "FILE_NOT_FOUND",
                "path": str(path),
            }

        if not new_code:
            return{
                "status": "error",
                "error_code": "NEW_CODE_REQUIRED",
                "path": str(path),
            }

        try:
            with open(path, "a", encoding="utf-8") as f:
                f.write("\n\n" + new_code)
            return{
                "status": "modified",
                "operation": "append",
                "path": str(path),
                "changes_summary": "Successfully appended new code"
            }
        except (PermissionError, OSError) as error:
            return{
                "status": "error",
                "error_code": "FILE_APPEND_FAILED",
                "path": str(path),
                "message": str(error)
            }

    #Write Operation
    elif operation == "write":
        if path.exists() and not overwrite:
            return{
                "status": "error",
                "error_code": "FILE_ALREADY_EXISTS",
                "path": str(path),
                "message": "File already exists at this path. Use overwrite=True if you really want to replace it, or use edit_file/append_to_file to modify it instead."
            }

        if new_code is None:
            return{
                "status": "error",
                "error_code": "NEW_CODE_REQUIRED",
                "path": str(path),
            }
        try:
            path.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            path.write_text(
                new_code,
                encoding="utf-8"
            )

            return{
                "status": "created",
                "operation": "write",
                "path": str(path)
            }
        
        except (PermissionError, OSError) as error:
            return{
                "status": "error",
                "error_code": "FILE_WRITE_FAILED",
                "path": str(path),
                "message": str(error)
            }
    return{
        "status": "error",
        "error_code": "INVALID_OPERATION",
        "operation": operation,
        "message": "Operations must be one of: read, edit, append, or write"
    }


In [20]:
#Constructing Agent and memory
llm=ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)
memory=MemorySaver()



In [11]:
#Deleting memory history/cache
# memory.storage.pop("test-8", None)

In [21]:
#Trimming users messages to preserve context window
@wrap_model_call
def limit_history(request, handler):
    messages = request.state["messages"]

    recent_messages = messages[-2:]

    request.state["messages"] = recent_messages

    return handler(request)

In [28]:
agent= create_agent(
    model=llm,
    tools=[retrieve_django_content, setup_django_project, manage_server,file_operations, search_codebase, run_django_commands],
    system_prompt = r"""
You are a Django engineering assistant with two jobs: reviewing Django code, and setting up/running Django projects using the tools available to you.
You have three categories of tools:
1. KNOWLEDGE TOOL — retrieve_django_content
   Call this FIRST for any of these:
   - User asks a question about Django concepts, patterns, debugging, or best practices
   - User wants a code review or explanation of existing code
   - User asks why something isn't working (debugging requires technical grounding)
   - User wants to understand an error message or Django behavior
   Do NOT call this for pure action requests: "create project X," "start the server," "stop the server" — these only need tool execution, not documentation.
   If a request mixes both (e.g. "review this code AND create a project"), call retrieve_django_content only for the review part.

2. FILE TOOLS — for reading and modifying code that already exists on disk:
   - search_codebase(query, search_scope="files"): Use this FIRST when the user refers to existing code but doesn't give an exact file path (e.g. "fix the bug in my views," "where is ALLOWED_HOSTS"). Returns file paths and line numbers. Do not read_file without this first.
   - search_codebase(query, search_scope="folders"): Use when searching for folder structure or organization of existing code.
   - file_operations(file_path, operation="read"): ALWAYS call this before editing or appending to any file. Never assume content — it may have changed.
   - file_operations(file_path, operation="edit", old_code, new_code): Replace an exact block of code. old_code must match EXACTLY from the read result, including whitespace. If not found or non-unique, re-read and adjust — do not retry blindly.
   - file_operations(file_path, operation="append", new_code): Add new code to the end of an existing file (functions, classes, utilities). Do NOT use for files with strict structure (urls.py, settings.py) — use edit instead.
   - file_operations(file_path, operation="write", new_code, overwrite=False): Create a new file only if it doesn't exist. If it exists, switch to edit or append — do not use overwrite=True unless the user explicitly asked to replace the entire file.

   WORKFLOW FOR CODE CHANGES:
   1. If you don't have the exact file path, call search_codebase first.
   2. Call file_operations with operation="read" to see current content.
   3. Decide: new file (write), new code added (append), or change existing (edit)?
   4. Make the change. Report in plain words what changed — do not paste raw tool output.
   5. If a step fails, re-read and adjust — do not guess or retry blindly.

3. ACTION TOOLS — these perform real actions on the user's machine:
   - setup_django_project(project_name, app_name, directory): Bootstraps a complete Django setup in one step (creates virtualenv, installs Django, initializes project, and generates specified apps).
   - manage_server(action, project_name, directory): Controls the Django development server. action="start" or action="stop".

   DIRECTORY RULE:
   - If the user's request doesn't specify a directory, leave the `directory` parameter out of the tool call entirely — do not fill it with "." or guess. The tool will use the correct default.
   - Never invent a directory path on your own when none is given.

STRICT ORDER OF OPERATIONS (for scaffolding requests):
1. setup_django_project creates virtualenv, Django, and the project in one step.
2. manage_server to start/stop as requested.
Skip any step only if the user says it already exists.

RULES FOR ACTION TOOLS:
- If the user asks to create, set up, scaffold, start, or run a Django project or app — use the tools. Do not just describe steps in text; call the tools.
- Infer sensible defaults if vague (e.g. project name "myproject") and state the assumption in one short sentence.
- After calling a tool, report what actually happened in one or two sentences — summarize, don't paste raw tool output.
- If a tool fails or the server doesn't start, say so plainly and suggest the likely cause — do not pretend it succeeded.
- Never call manage_server(action="start") without confirming the project exists at that path.
- Only call manage_server(action="stop") if the user asks to stop/restart, or if you're about to start a new server and one may already be running.
- If the user only asks you to explain what steps you'd take, describe them in words — do not call tools.

WORKFLOW FOR CODE REVIEWS / KNOWLEDGE ANSWERS:
1. Call retrieve_django_content with the user's question or code snippet.
2. Answer using what the retriever returns. If it's empty or irrelevant, say "I don't have this information in my knowledge base" — do not guess.
3. Keep answers direct: lead with the answer, then 1-3 sentences of context. Use bullet points only if the user explicitly asks for a list.
4. Do NOT narrate tool calls (no "I searched the knowledge base..." or "I'm now calling retrieve_django_content...") — just do it and report the outcome.

DJANGO COMMANDS TOOL — run_commands:
Call this when the user asks to run Django management commands on their project.
Supported commands: makemigrations, migrate, collectstatic, test, flush, check.

Usage:
- run_commands(command="makemigrations", project_name="myproject") — detect schema changes
- run_commands(command="makemigrations", project_name="myproject", app_label="accounts") — for a specific app
- run_commands(command="migrate", project_name="myproject") — apply migrations to database
- run_commands(command="migrate", project_name="myproject", app_label="accounts") — migrate specific app
- run_commands(command="collectstatic", project_name="myproject", no_input=True) — collect static files
- run_commands(command="test", project_name="myproject") — run all tests
- run_commands(command="test", project_name="myproject", app_label="api", verbose=True) — run tests with verbose output
- run_commands(command="flush", project_name="myproject", no_input=True) — clear database (destructive)
- run_commands(command="check", project_name="myproject") — validate project configuration

After calling run_commands, report what the command did in one or two sentences — summarize, don't paste raw output.
If the command fails, say so plainly and show the error — do not pretend it succeeded.

GENERAL:
- Be concise. No filler.
- If a request is ambiguous between "explain how to do X" and "do X for me," default to doing it if action tools fit — ask only if ambiguity would act on the wrong project/directory.
""",
    middleware=[limit_history],
    checkpointer=memory
)

#Memory
config = {"configurable": {"thread_id": "test-26"}}

In [29]:
#Creatin User Interface
while True:
    question= input("Any Question about django: ").strip()
    if question.lower() in ["exit", "quit", "q"]:
        print("See you soon...")
        break
    if not question:
        continue

    result= agent.invoke({
        "messages": [HumanMessage(content=question)]
    }, config=config)

    print("\n Final Result")
    print(result["messages"][-1].content)


 Final Result
The Django development server for **blog_platform** has been started successfully. Let me know if you need anything else (e.g., running migrations, creating an app, or stopping the server).
Enabling codebase search...
Searching file contents...
Enabling codebase search...
Searching file contents...
Performing 'read' on C:\Users\ZENOID\Desktop\Home\home\self_made.projects\AI(created by me) generated projects\blog_platform\blog_platform\urls.py.......
Enabling codebase search...
Searching file contents...
Enabling codebase search...
Searching folders...
Performing 'read' on C:\Users\ZENOID\Desktop\Home\home\self_made.projects\AI(created by me) generated projects\blog_platform\posts\urls.py.......

 Final Result
Your project’s URL configuration is:

**Root URLconf (`blog_platform/urls.py`)**
```text
/
├─ admin/          → Django admin site
└─ api/            → Includes the URLs from the `posts` app
```

**Posts app URLs (`posts/urls.py`)**
```text
api/posts/        → `PostL

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0azq1pfenatsrbgfy8g0cac` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 196180, Requested 7378. Please try again in 25m37.056s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}